**Knowledge Engineering**
**: Self-Driving Trailer Agent**


**Step 1: Identify the Task**

The task is to build a knowledge-based agent that controls a self-driving trailer hauling cargo from Nairobi to Kisumu.

**Goal:** Deliver the cargo safely, making driving decisions that minimise risk to the trailer, its load, and other road users.


**Questions the agent must be able to answer:**
1.  Given the current traffic light, what should the trailer do?
2.  Given the current road condition (rain, fog, steep descent, and so on), how should the trailer adjust its driving?
3.  Is there an obstacle ahead, and if so, how should the trailer respond?
4.  Is the trailer in a fit state to depart (for example, is it overloaded)?
5.  Which towns are directly connected, and how far apart are they?
6.  When several percepts apply at once, which rule takes priority?
7.  What is the shortest route between two towns, and how long is it?

**Step 2: Assemble Relevant Knowledge**

The agent needs the following categories of knowledge:

| # | Knowledge category | What the agent needs to know | Source |
|---|---|---|---|
| 1 | Road network | The towns on the corridor, which towns are directly connected by road, and the distance (km) of each road segment | Practical Exercise 3 |
| 2 | Traffic signals | The possible traffic light states and the correct action for each | Basic road rules |
| 3 | Road conditions | Conditions that affect a heavy vehicle (wet road, fog, steep descent, potholes, road works, clear road) and the safe response to each | Domain knowledge about operating heavy vehicles |
| 4 | Obstacles | Objects that may appear in the trailer's path (pedestrian, animal, stalled vehicle) and the safe response to each | Domain knowledge |
| 5 | Load status | Whether the trailer is empty, normally loaded, or overloaded, and how this affects driving and departure | Domain knowledge |
| 6 | Rule priority | The order in which rules apply when more than one percept is active (for example, an obstacle overrides a green light) | Design decision for safe operation |
| 7 | Route selection | A procedure for choosing the cheapest (shortest) sequence of roads from the start town to the destination | Uniform Cost Search, adapted from practical exercise 3 |



**Step 3: Define the Vocabulary**


**Predicates**

| Predicate | Arity | Meaning |
|---|---|---|
| `Town(x)` | 1 | x is a town on the corridor |
| `Road(x, y, d)` | 3 | There is a direct road from town x to town y of length d km |
| `Light(c)` | 1 | The traffic light ahead currently shows signal c |
| `Condition(s)` | 1 | The road ahead currently has condition s |
| `Obstacle(o)` | 1 | Object o is in the trailer's path |
| `Load(l)` | 1 | The trailer's current load status is l |
| `Action(a)` | 1 | The agent should perform action a |
| `CanDepart()` | 0 | The trailer is permitted to begin a trip |
| `Reachable(x, y, p, k)` | 4 | Town y can be reached from town x along path p, with total length k km |
| `Route(x, y, p, k)` | 4 | p is the cheapest path from town x to town y, with total length k km |

**Functions**

| Function | Meaning |
|---|---|
| `Distance(x, y)` | Returns d where `Road(x, y, d)` holds |
| `ShortestRoute(x, y)` | Returns (k, p) where `Route(x, y, p, k)` holds, found by UCS |

**Constants**

| Type | Constants |
|---|---|
| Towns | `nairobi`, `limuru`, `mai_mahiu`, `naivasha`, `nakuru`, `eldama_ravine`, `kericho`, `eldoret`, `kapsabet`, `awasi`, `kisumu` |
| Light signals | `Red`, `Yellow`, `Green`, `Flashing_Red`, `Flashing_Yellow`, `Off` |
| Road conditions | `Clear`, `Wet`, `Fog`, `Steep_Descent`, `Potholes`, `Road_Works` |
| Obstacles | `None`, `Pedestrian`, `Animal`, `Stalled_Vehicle`, `Debris` |
| Load statuses | `Empty`, `Loaded`, `Overloaded` |

**General knowledge as logical rules**

These are the rules Step 4 encodes in Python:

1.  `Light(Red) ⇒ Action(Stop)`
2.  `Light(Yellow) ⇒ Action(SlowDown)`
3.  `Light(Green) ∧ Obstacle(None) ⇒ Action(Proceed)`
4.  `Light(Off) ⇒ Action(TreatAsAllWayStop)`
5.  `∀o [Obstacle(o) ∧ o ≠ None ⇒ Action(Stop)]` (obstacle response overrides any light)
6.  `Condition(Steep_Descent) ∧ (Load(Loaded) ∨ Load(Overloaded)) ⇒ Action(EngageLowGear)`
7.  `Condition(Wet) ∨ Condition(Fog) ⇒ Action(ReduceSpeed)`
8.  `Load(Overloaded) ⇒ ¬CanDepart()`
9.  `∀x ∀y ∀d [Road(x, y, d) ⇒ Town(x) ∧ Town(y) ∧ d > 0]`
10. `∀x [Town(x) ⇒ Route(x, x, [x], 0)]` (a town is reachable from itself at zero cost)
11. `∀x ∀y ∀z ∀p ∀k ∀d [Route(x, y, p, k) ∧ Road(y, z, d) ⇒ Reachable(x, z, p + [z], k + d)]`; `Route` selects the reachable path with the minimum total length

**Rule priority (highest first):** Obstacle → Load → Light → Condition. A higher-priority rule overrides a lower one when they disagree.

**Step 4: Encode General Knowledge about the Domain**

The vocabulary from Step 3 is now encoded in Python. Each predicate becomes a dictionary so that the agent can look up the correct action in constant time.

**4.1 Road network facts: `Town(x)` and `Road(x, y, d)`**

These are the same road facts used by the UCS search in exercise 3.

In [ ]:
# Road(x, y, d): a direct road from town x to town y of length d km
# Source: UCS_Trailer.ipynb
town_graph = {
    "nairobi":       {"limuru": 38},
    "limuru":        {"mai_mahiu": 20, "naivasha": 55},
    "mai_mahiu":     {"naivasha": 35},
    "naivasha":      {"nakuru": 71},
    "nakuru":        {"eldama_ravine": 64, "kericho": 115, "eldoret": 156},
    "eldama_ravine": {"eldoret": 108},
    "kericho":       {"awasi": 43},
    "eldoret":       {"kapsabet": 46},
    "kapsabet":      {"awasi": 56},
    "awasi":         {"kisumu": 41},
    "kisumu":        {},
}

# Town(x): every town that appears in the network
towns = set(town_graph)

def distance(x, y):
    """Distance(x, y): return d if Road(x, y, d) holds, otherwise None."""
    return town_graph.get(x, {}).get(y)

**4.2 Traffic light rules: `Light(c) ⇒ Action(a)`**

In [ ]:
light_rules = {
    "Red":             "Stop: it is unsafe to move.",
    "Yellow":          "Slow down: prepare to stop (a loaded trailer needs a longer stopping distance).",
    "Green":           "Proceed: it is safe to move if the path is clear.",
    "Flashing_Red":    "Stop fully, then proceed only when the junction is clear.",
    "Flashing_Yellow": "Proceed slowly with caution.",
    "Off":             "Signal failure: treat the junction as an all-way stop.",
}

**4.3 Road condition rules: `Condition(s) ⇒ Action(a)`**

In [ ]:
condition_rules = {
    "Clear":         "Maintain normal cruising speed.",
    "Wet":           "Reduce speed and increase following distance.",
    "Fog":           "Reduce speed, switch on fog lights and hazard lights.",
    "Steep_Descent": "Engage low gear and use engine braking to avoid brake overheating.",
    "Potholes":      "Reduce speed and steer around potholes where safe.",
    "Road_Works":    "Reduce speed and follow diversion signs.",
}

**4.4 Obstacle rules: `Obstacle(o) ⇒ Action(a)`**

In [ ]:
obstacle_rules = {
    "None":            "Path is clear: continue.",
    "Pedestrian":      "Stop immediately and wait for the pedestrian to cross.",
    "Animal":          "Stop and wait for the animal to clear the road.",
    "Stalled_Vehicle": "Stop, switch on hazard lights, and overtake only when the opposite lane is clear.",
    "Debris":          "Slow down and stop if the debris cannot be safely avoided.",
}

**4.5 Load rules: `Load(l) ⇒ Action(a)` and `CanDepart()`**

In [ ]:
load_rules = {
    "Empty":      "Normal driving: the trailer is light and can brake quickly.",
    "Loaded":     "Allow longer stopping distances and use low gear on descents.",
    "Overloaded": "Do not depart: offload excess cargo before the trip.",
}

# CanDepart(): False whenever Load(Overloaded) holds (Rule 8)
can_depart = {"Empty": True, "Loaded": True, "Overloaded": False}

**4.6 Rule priority**

When several percepts are active at once, the agent consults the rule sets in this order and the first one that demands caution wins. This encodes the safety principle from Step 1.

In [ ]:
# Highest priority first
rule_priority = [
    ("Obstacle",  obstacle_rules),
    ("Load",      load_rules),
    ("Light",     light_rules),
    ("Condition", condition_rules),
]

# Combined knowledge base: every rule set, grouped by predicate
knowledge_base = {name: rules for name, rules in rule_priority}
knowledge_base["Road"] = town_graph

**4.7 Route selection: `ShortestRoute(x, y)` by Uniform Cost Search**

Rules 10 and 11 define a route as a chain of `Road` facts. Many such chains can exist, so the agent needs a procedure to pick the cheapest one. Uniform Cost Search does this by always expanding the partial route with the lowest total distance so far. Because every road length is positive (Rule 9), the first time the destination is removed from the frontier, its route is guaranteed to be the shortest.

The search reads directly from `town_graph`, so any road fact added to the knowledge base later is automatically considered when selecting a route.

In [ ]:
import heapq

def shortest_route(start, goal, graph=town_graph, verbose=False):
    """ShortestRoute(x, y): return (k, p) such that Route(x, y, p, k) holds.

    Returns (None, []) if either town is unknown or no route exists.
    """
    if start not in graph or goal not in graph:
        return None, []

    frontier = [(0, start, [start])]   # (cost so far, town, path); Rule 10
    explored = set()

    while frontier:
        cost, town, path = heapq.heappop(frontier)   # cheapest partial route
        if verbose:
            print(f"Popped: {town} (cost {cost})")

        if town == goal:
            return cost, path

        if town in explored:
            continue
        explored.add(town)

        for neighbour, d in graph[town].items():      # Rule 11: extend by Road(town, neighbour, d)
            if neighbour not in explored:
                heapq.heappush(frontier, (cost + d, neighbour, path + [neighbour]))
                if verbose:
                    print(f"  Queued: {neighbour} (cost {cost + d})")

    return None, []

# Add the route procedure to the combined knowledge base
knowledge_base["Route"] = shortest_route

**4.8 Display the encoded knowledge base**

In [ ]:
def show_kb(start="nairobi", goal="kisumu"):
    print("=== Self-Driving Trailer Knowledge Base ===")
    for name, rules in rule_priority:
        print(f"\n--- {name} rules ---")
        for percept, action in rules.items():
            print(f"{percept:16} ➜ {action}")

    print("\n--- Road facts: Road(x, y, d) ---")
    for x, neighbours in town_graph.items():
        for y, d in neighbours.items():
            print(f"Road({x}, {y}, {d} km)")

    print("\n--- Rule priority (highest first) ---")
    print(" > ".join(name for name, _ in rule_priority))

    cost, path = shortest_route(start, goal)
    print(f"\n--- Route selection: ShortestRoute({start}, {goal}) ---")
    if path:
        print("Route:", " -> ".join(path))
        print(f"Total distance: {cost} km")
    else:
        print("No route found.")

    total_rules = sum(len(r) for _, r in rule_priority)
    total_roads = sum(len(n) for n in town_graph.values())
    print(f"\nTotal: {total_rules} driving rules, {len(towns)} towns, {total_roads} road facts")

show_kb()

=== Self-Driving Trailer Knowledge Base ===

--- Obstacle rules ---
None             ➜ Path is clear: continue.
Pedestrian       ➜ Stop immediately and wait for the pedestrian to cross.
Animal           ➜ Stop and wait for the animal to clear the road.
Stalled_Vehicle  ➜ Stop, switch on hazard lights, and overtake only when the opposite lane is clear.
Debris           ➜ Slow down and stop if the debris cannot be safely avoided.

--- Load rules ---
Empty            ➜ Normal driving: the trailer is light and can brake quickly.
Loaded           ➜ Allow longer stopping distances and use low gear on descents.
Overloaded       ➜ Do not depart: offload excess cargo before the trip.

--- Light rules ---
Red              ➜ Stop: it is unsafe to move.
Yellow           ➜ Slow down: prepare to stop (a loaded trailer needs a longer stopping distance).
Green            ➜ Proceed: it is safe to move if the path is clear.
Flashing_Red     ➜ Stop fully, then proceed only when the junction is clear.
Fla

**Steps 5-7: Methods for adding new facts and posing queries to the knowledge base**

**5: Methods for adding new facts to the knowledge base**

In [ ]:
def add_rule(kind, key, action, overwrite=False, depart_allowed=None):
    kind_norm = _normalise(kind)
    if kind_norm not in KIND_TO_RULESET:
        return _result("rejected", kind, key, None,
                       f"Unknown rule type '{kind}'. Valid types: {', '.join(KIND_TO_RULESET)}.")
    if key is None or not str(key).strip() or action is None or not str(action).strip():
        return _result("rejected", kind_norm, key, None, "Both a name and an action are required.")

    rules = knowledge_base[KIND_TO_RULESET[kind_norm]]
    existing = _find_key(rules, key)

    if existing is not None and not overwrite:
        return _result("rejected", kind_norm, existing, rules[existing],
                       f"'{existing}' already exists. Use overwrite=True to replace it.")

    if kind_norm == "load" and existing is None and not isinstance(depart_allowed, bool):
        return _result("rejected", kind_norm, key, None,
                       "A new load status needs depart_allowed=True or False.")

    canonical = existing or str(key).strip().replace(" ", "_")
    rules[canonical] = str(action).strip()
    if kind_norm == "load" and isinstance(depart_allowed, bool):
        can_depart[canonical] = depart_allowed

    status = "updated" if existing else "added"
    return _result(status, kind_norm, canonical, rules[canonical],
                   f"{kind_norm.capitalize()} rule '{canonical}' {status}.")

def add_road(x, y, d, bidirectional=False):
    if not x or not y or not str(x).strip() or not str(y).strip():
        return _result("rejected", "road", (x, y), None, "Both town names are required.")
    if isinstance(d, bool) or not isinstance(d, (int, float)) or d <= 0:
        return _result("rejected", "road", (x, y), None,
                       "Distance must be a positive number (Rule 9).")

    graph = knowledge_base["Road"]
    cx = _find_key(graph, x) or str(x).strip().lower().replace(" ", "_")
    cy = _find_key(graph, y) or str(y).strip().lower().replace(" ", "_")
    if cx == cy:
        return _result("rejected", "road", (cx, cy), None, "A road must connect two different towns.")

    pairs = [(cx, cy)] + ([(cy, cx)] if bidirectional else [])
    changes = []
    for a, b in pairs:
        graph.setdefault(a, {})
        graph.setdefault(b, {})
        old = graph[a].get(b)
        graph[a][b] = d
        changes.append(f"Road({a}, {b}, {d} km) " + ("updated" if old is not None else "added"))

    towns.update(graph)
    return _result("added", "road", (cx, cy), d, "; ".join(changes) + ".")

**6-7: Methods for querying the knowledge base**


In [ ]:
# Fallbacks used when a percept is not in the knowledge base
DEFAULT_ACTIONS = {
    "light":     "Unknown signal: treat the junction as an all-way stop and request human assistance.",
    "condition": "Unknown road condition: reduce speed and proceed with caution.",
    "obstacle":  "Unknown obstacle: stop and request human assistance.",
    "load":      "Unknown load status: do not depart until the load is verified.",
}

KIND_TO_RULESET = {
    "light":     "Light",
    "condition": "Condition",
    "obstacle":  "Obstacle",
    "load":      "Load",
}

def _normalise(text):
    return str(text).strip().lower().replace("-", "_").replace(" ", "_")

def _find_key(rules, key):
    wanted = _normalise(key)
    for canonical in rules:
        if _normalise(canonical) == wanted:
            return canonical
    return None

def _result(status, kind, key, answer, message):
    return {"status": status, "kind": kind, "key": key,
            "answer": answer, "message": message}

def query(kind, key):
    kind_norm = _normalise(kind)
    if kind_norm not in KIND_TO_RULESET:
        valid = ", ".join(KIND_TO_RULESET)
        return _result("invalid", kind, key, None,
                       f"Unknown query type '{kind}'. Valid types: {valid}.")

    # Python's None means "no obstacle" for obstacle queries
    if key is None and kind_norm == "obstacle":
        key = "None"

    if key is None or str(key).strip() == "":
        return _result("invalid", kind_norm, key, None,
                       "No value supplied to look up.")

    rules = knowledge_base[KIND_TO_RULESET[kind_norm]]
    canonical = _find_key(rules, key)

    if canonical is not None:
        return _result("found", kind_norm, canonical, rules[canonical],
                       f"'{canonical}' found in the {kind_norm} rules.")

    return _result("unknown", kind_norm, key, DEFAULT_ACTIONS[kind_norm],
                   f"'{key}' is not in the knowledge base for {kind_norm}. "
                   f"Applying the safe default.")

def query_can_depart(load_status):
    # Is the trailer permitted to depart, given its load status?
    canonical = _find_key(can_depart, load_status) if load_status is not None else None
    if canonical is None:
        return _result("unknown", "can_depart", load_status, False,
                       f"Load status '{load_status}' is not in the knowledge base. "
                       f"Departure is NOT permitted until the load is verified.")
    allowed = can_depart[canonical]
    return _result("found", "can_depart", canonical, allowed,
                   f"Load '{canonical}': departure {'permitted' if allowed else 'not permitted'}.")

def query_distance(x, y):
    graph = knowledge_base["Road"]
    cx, cy = _find_key(graph, x), _find_key(graph, y)

    for name, canon in ((x, cx), (y, cy)):
        if canon is None:
            return _result("unknown", "distance", (x, y), None,
                           f"Town '{name}' is not in the knowledge base.")

    d = graph[cx].get(cy)
    if d is None:
        return _result("unknown", "distance", (cx, cy), None,
                       f"No direct road from {cx} to {cy} is recorded.")
    return _result("found", "distance", (cx, cy), d,
                   f"Road({cx}, {cy}, {d} km).")

def query_route(start, goal):
    graph = knowledge_base["Road"]
    cs, cg = _find_key(graph, start), _find_key(graph, goal)

    for name, canon in ((start, cs), (goal, cg)):
        if canon is None:
            return _result("unknown", "route", (start, goal), None,
                           f"Town '{name}' is not in the knowledge base.")

    cost, path = knowledge_base["Route"](cs, cg)
    if not path:
        return _result("unknown", "route", (cs, cg), None,
                       f"No route from {cs} to {cg} exists in the knowledge base.")
    return _result("found", "route", (cs, cg), {"path": path, "distance_km": cost},
                   f"Shortest route: {' -> '.join(path)} ({cost} km).")

def show(result):
    print(f"[{result['status'].upper()}] {result['message']}")
    if result["answer"] is not None:
        print(f"   Answer: {result['answer']}")

**Step 8: Querying the Knowledge Base**

The agent now poses the questions identified in Step 1 to the knowledge base, using the query methods from 4.9. Each answer comes either from a stored fact (`FOUND`) or from a safe default when the knowledge base has no matching fact (`UNKNOWN`).

**8.1 Single-percept queries (Step 1, questions 1 to 5 and 7)**

In [ ]:
print("Q1. Light is Yellow: what should the trailer do?")
show(query("light", "Yellow"))

print("\nQ2. Road ahead is a steep descent: how should the trailer adjust?")
show(query("condition", "Steep_Descent"))

print("\nQ3. A stalled vehicle is ahead: how should the trailer respond?")
show(query("obstacle", "Stalled_Vehicle"))

print("\nQ4. Can the trailer depart when Loaded? When Overloaded?")
show(query_can_depart("Loaded"))
show(query_can_depart("Overloaded"))

print("\nQ5. How far is Nakuru from Kericho by direct road?")
show(query_distance("nakuru", "kericho"))

print("\nQ7. What is the shortest route from Nairobi to Kisumu?")
show(query_route("nairobi", "kisumu"))

Q1. Light is Yellow: what should the trailer do?
[FOUND] 'Yellow' found in the light rules.
   Answer: Slow down: prepare to stop (a loaded trailer needs a longer stopping distance).

Q2. Road ahead is a steep descent: how should the trailer adjust?
[FOUND] 'Steep_Descent' found in the condition rules.
   Answer: Engage low gear and use engine braking to avoid brake overheating.

Q3. A stalled vehicle is ahead: how should the trailer respond?
[FOUND] 'Stalled_Vehicle' found in the obstacle rules.
   Answer: Stop, switch on hazard lights, and overtake only when the opposite lane is clear.

Q4. Can the trailer depart when Loaded? When Overloaded?
[FOUND] Load 'Loaded': departure permitted.
   Answer: True
[FOUND] Load 'Overloaded': departure not permitted.
   Answer: False

Q5. How far is Nakuru from Kericho by direct road?
[FOUND] Road(nakuru, kericho, 115 km).
   Answer: 115

Q7. What is the shortest route from Nairobi to Kisumu?
[FOUND] Shortest route: nairobi -> limuru -> mai_mahiu -

**8.2 Combined percepts and rule priority (Step 1, question 6)**

In practice the trailer perceives several things at once. The rule set in 4.6 (`rule_priority`) gives the order Obstacle > Load > Light > Condition, but the single-percept queries above never apply it. The `decide` function below does: it queries every percept, marks each answer as either routine (for example a green light or a clear road) or requiring caution, and selects the highest-priority action that requires caution. The remaining cautionary actions are kept as advisories, so the agent still slows down on a wet road even while obeying a red light.

Any percept that is unknown or invalid is treated as requiring caution, because the agent cannot confirm that it is safe.

In [ ]:
# Percept values that need no special response
ROUTINE = {
    "Obstacle":  {"None"},
    "Load":      {"Empty", "Loaded"},
    "Light":     {"Green"},
    "Condition": {"Clear"},
}

def decide(percepts):
    """Apply rule_priority to a set of simultaneous percepts.

    percepts: dict such as {"light": "Green", "obstacle": "Pedestrian"}
    Returns the deciding rule, the chosen action, and any advisories.
    """
    cautions = []
    for name, _ in rule_priority:                        # highest priority first
        kind = name.lower()
        if kind not in percepts:
            continue
        r = query(kind, percepts[kind])
        if r["status"] == "found":
            routine = r["key"] in ROUTINE[name]
            action = r["answer"]
        else:                                            # unknown or invalid percept
            routine = False
            action = DEFAULT_ACTIONS[kind]
        if not routine:
            cautions.append((name, r["key"], action))

    if not cautions:
        return {"rule": None, "decision": "All percepts routine: proceed normally.", "advisories": []}
    (name, key, action), rest = cautions[0], cautions[1:]
    return {"rule": f"{name}({key})", "decision": action,
            "advisories": [f"{n}({k}): {a}" for n, k, a in rest]}

def show_decision(percepts):
    d = decide(percepts)
    print("Percepts:", percepts)
    print(f"  Deciding rule: {d['rule']}")
    print(f"  Decision:      {d['decision']}")
    for a in d["advisories"]:
        print(f"  Advisory:      {a}")
    print()

print("Q6. Which rule takes priority when several percepts apply?\n")
show_decision({"light": "Green", "obstacle": "None", "condition": "Clear", "load": "Loaded"})
show_decision({"light": "Green", "obstacle": "Pedestrian"})                     # Rule 5 overrides Rule 3
show_decision({"light": "Red", "condition": "Wet"})                             # Rule 1 plus Rule 7 advisory
show_decision({"light": "Green", "condition": "Steep_Descent", "load": "Loaded"})  # Rule 6
show_decision({"light": "Green", "load": "Overloaded"})                         # Rule 8

Q6. Which rule takes priority when several percepts apply?

Percepts: {'light': 'Green', 'obstacle': 'None', 'condition': 'Clear', 'load': 'Loaded'}
  Deciding rule: None
  Decision:      All percepts routine: proceed normally.

Percepts: {'light': 'Green', 'obstacle': 'Pedestrian'}
  Deciding rule: Obstacle(Pedestrian)
  Decision:      Stop immediately and wait for the pedestrian to cross.

Percepts: {'light': 'Red', 'condition': 'Wet'}
  Deciding rule: Light(Red)
  Decision:      Stop: it is unsafe to move.
  Advisory:      Condition(Wet): Reduce speed and increase following distance.

Percepts: {'light': 'Green', 'condition': 'Steep_Descent', 'load': 'Loaded'}
  Deciding rule: Condition(Steep_Descent)
  Decision:      Engage low gear and use engine braking to avoid brake overheating.

Percepts: {'light': 'Green', 'load': 'Overloaded'}
  Deciding rule: Load(Overloaded)
  Decision:      Do not depart: offload excess cargo before the trip.



**8.3 A complete trip query**

This combines route selection with the driving rules: before departure the agent checks `CanDepart()`, then plans the route, then reacts to the percepts reported on each road segment. The percepts below are a hypothetical scenario for demonstration, not real road data.

In [ ]:
def plan_trip(start, goal, load_status, segment_percepts):
    print(f"=== Trip: {start} to {goal}, load = {load_status} ===")
    dep = query_can_depart(load_status)
    show(dep)
    if dep["answer"] is not True:
        print("Trip cancelled.")
        return

    route = query_route(start, goal)
    show(route)
    if route["status"] != "found":
        return

    path = route["answer"]["path"]
    for a, b in zip(path, path[1:]):
        percepts = dict(segment_percepts.get((a, b), {}), load=load_status)
        d = decide(percepts)
        print(f"\n{a} -> {b} ({distance(a, b)} km): {d['decision']}")
        for adv in d["advisories"]:
            print(f"    Advisory: {adv}")

# Hypothetical percepts on some segments of the route
scenario = {
    ("limuru", "mai_mahiu"): {"condition": "Steep_Descent"},
    ("naivasha", "nakuru"):  {"light": "Red", "condition": "Wet"},
    ("nakuru", "kericho"):   {"condition": "Fog", "obstacle": "Animal"},
    ("awasi", "kisumu"):     {"light": "Green", "condition": "Road_Works"},
}
plan_trip("nairobi", "kisumu", "Loaded", scenario)
print()
plan_trip("nairobi", "kisumu", "Overloaded", scenario)

=== Trip: nairobi to kisumu, load = Loaded ===
[FOUND] Load 'Loaded': departure permitted.
   Answer: True
[FOUND] Shortest route: nairobi -> limuru -> mai_mahiu -> naivasha -> nakuru -> kericho -> awasi -> kisumu (363 km).
   Answer: {'path': ['nairobi', 'limuru', 'mai_mahiu', 'naivasha', 'nakuru', 'kericho', 'awasi', 'kisumu'], 'distance_km': 363}

nairobi -> limuru (38 km): All percepts routine: proceed normally.

limuru -> mai_mahiu (20 km): Engage low gear and use engine braking to avoid brake overheating.

mai_mahiu -> naivasha (35 km): All percepts routine: proceed normally.

naivasha -> nakuru (71 km): Stop: it is unsafe to move.
    Advisory: Condition(Wet): Reduce speed and increase following distance.

nakuru -> kericho (115 km): Stop and wait for the animal to clear the road.
    Advisory: Condition(Fog): Reduce speed, switch on fog lights and hazard lights.

kericho -> awasi (43 km): All percepts routine: proceed normally.

awasi -> kisumu (41 km): Reduce speed and follow 

**Debugging the Knowledge Base**

Debugging is done in three parts:
1.  **Consistency checks** confirm that the encoded facts obey the rules declared in Step 3.
2.  **A test suite** runs each query and update method against normal, edge-case and faulty inputs, and compares the result with the expected behaviour.
3.  **Fixes** are applied for every failure, and the suite is run again.

Several tests add facts to the knowledge base, so its state is saved first and restored after each run. This keeps the tests from polluting the real knowledge base.

**9.1 Snapshot and restore helpers**

In [ ]:
import copy, math

def snapshot():
    state = {k: copy.deepcopy(v) for k, v in knowledge_base.items() if isinstance(v, dict)}
    state["_can_depart"] = copy.deepcopy(can_depart)
    return state

def restore(state):
    for k, v in state.items():
        target = can_depart if k == "_can_depart" else knowledge_base[k]
        target.clear()
        target.update(copy.deepcopy(v))
    towns.clear()
    towns.update(knowledge_base["Road"])

clean_state = snapshot()

**9.2 Consistency checks against the Step 3 rules**

In [ ]:
VOCABULARY = {
    "Light":     {"Red", "Yellow", "Green", "Flashing_Red", "Flashing_Yellow", "Off"},
    "Condition": {"Clear", "Wet", "Fog", "Steep_Descent", "Potholes", "Road_Works"},
    "Obstacle":  {"None", "Pedestrian", "Animal", "Stalled_Vehicle", "Debris"},
    "Load":      {"Empty", "Loaded", "Overloaded"},
}

def consistency_checks():
    problems = []
    graph = knowledge_base["Road"]

    # Rule 9: every road joins two known towns and has a positive, finite length
    for x, nbrs in graph.items():
        for y, d in nbrs.items():
            if y not in towns:
                problems.append(f"Road({x}, {y}): {y} is not a Town")
            if isinstance(d, bool) or not isinstance(d, (int, float)) or not math.isfinite(d) or d <= 0:
                problems.append(f"Road({x}, {y}, {d}): length violates Rule 9")

    # Every vocabulary constant has a rule, and every rule uses a declared constant
    for name, consts in VOCABULARY.items():
        rules = knowledge_base[name]
        for c in consts - set(rules):
            problems.append(f"{name}({c}) is in the vocabulary but has no rule")
        for c in set(rules) - consts:
            problems.append(f"{name}({c}) has a rule but is not in the vocabulary")

    # Rule 8: every load status has a CanDepart value, and Overloaded cannot depart
    for l in knowledge_base["Load"]:
        if l not in can_depart:
            problems.append(f"Load({l}) has no CanDepart() value")
    if can_depart.get("Overloaded") is not False:
        problems.append("Rule 8 violated: Overloaded trailer may depart")

    # Every rule set appears exactly once in the priority order
    names = [n for n, _ in rule_priority]
    if sorted(names) != sorted(VOCABULARY):
        problems.append(f"rule_priority covers {names}, expected {sorted(VOCABULARY)}")

    # Every town can reach the destination (needed for a trailer to complete any trip)
    for t in sorted(towns):
        if t != "kisumu" and not shortest_route(t, "kisumu")[1]:
            problems.append(f"No route from {t} to kisumu")
    # The trailer must be able to return to its depot
    if not shortest_route("kisumu", "nairobi")[1]:
        problems.append("No return route from kisumu to nairobi (all roads are one-way)")

    print("--- Consistency checks ---")
    if problems:
        for p in problems:
            print("  PROBLEM:", p)
    else:
        print("  All checks passed.")
    return problems

_ = consistency_checks()

--- Consistency checks ---
  PROBLEM: No return route from kisumu to nairobi (all roads are one-way)


**9.3 Test suite**

Each test states the input, the expected behaviour, and whether the knowledge base produced it.

In [ ]:
def run_tests():
    tests = [
        # (description, function call, check on the result)
        # --- Normal queries ---
        ("Known light 'Red'",                 lambda: query("light", "Red"),                lambda r: r["status"] == "found"),
        ("Case and spacing ' red '",          lambda: query("light", " red "),              lambda r: r["key"] == "Red"),
        ("Space instead of underscore",       lambda: query("light", "Flashing red"),       lambda r: r["key"] == "Flashing_Red"),
        ("Route Nairobi to Kisumu is 363 km", lambda: query_route("nairobi", "kisumu"),     lambda r: r["answer"]["distance_km"] == 363),
        ("Route from a town to itself",       lambda: query_route("nakuru", "nakuru"),      lambda r: r["answer"]["distance_km"] == 0),
        # --- Faulty or unknown inputs must fall back to a safe default ---
        ("Unknown light 'Blue'",              lambda: query("light", "Blue"),               lambda r: r["status"] == "unknown" and "stop" in r["answer"].lower()),
        ("Non-string light value 123",        lambda: query("light", 123),                  lambda r: r["status"] == "unknown"),
        ("Python None as obstacle = no obstacle", lambda: query("obstacle", None),          lambda r: r["key"] == "None"),
        ("Unknown query type 'weather'",      lambda: query("weather", "Wet"),              lambda r: r["status"] == "invalid"),
        ("Unknown load cannot depart",        lambda: query_can_depart("Half_Full"),        lambda r: r["answer"] is False),
        ("Unknown town",                      lambda: query_route("nairobi", "mombasa"),    lambda r: r["status"] == "unknown"),
        ("Empty obstacle reading is not safe",lambda: decide({"obstacle": ""}),             lambda r: r["rule"] is not None),
        # --- Rule priority ---
        ("Obstacle overrides green light",    lambda: decide({"light": "Green", "obstacle": "Pedestrian"}), lambda r: r["rule"] == "Obstacle(Pedestrian)"),
        ("Overloaded overrides green light",  lambda: decide({"light": "Green", "load": "Overloaded"}),     lambda r: r["rule"] == "Load(Overloaded)"),
        # --- Return trip ---
        ("Return route Kisumu to Nairobi",    lambda: query_route("kisumu", "nairobi"),     lambda r: r["status"] == "found"),
        # --- Adding facts ---
        ("Reject duplicate rule",             lambda: add_rule("light", "red", "Go"),       lambda r: r["status"] == "rejected"),
        ("New load needs depart_allowed",     lambda: add_rule("load", "Half_Full", "x"),   lambda r: r["status"] == "rejected"),
        ("New rule name follows naming style",lambda: add_rule("light", "purple arrow", "Stop."), lambda r: r["key"] == "Purple_Arrow"),
        ("Reject zero distance",              lambda: add_road("a", "b", 0),                lambda r: r["status"] == "rejected"),
        ("Reject negative distance",          lambda: add_road("a", "b", -5),               lambda r: r["status"] == "rejected"),
        ("Reject text distance",              lambda: add_road("a", "b", "40"),             lambda r: r["status"] == "rejected"),
        ("Reject NaN distance",               lambda: add_road("a", "b", float("nan")),     lambda r: r["status"] == "rejected"),
        ("Reject infinite distance",          lambda: add_road("a", "b", float("inf")),     lambda r: r["status"] == "rejected"),
    ]

    failures = []
    print(f"{'Result':6}  Test")
    for desc, call, check in tests:
        try:
            r = call()
            ok = bool(check(r))
        except Exception as e:
            r, ok = f"{type(e).__name__}: {e}", False
        print(f"{'PASS' if ok else 'FAIL':6}  {desc}")
        if not ok:
            failures.append((desc, r))

    restore(clean_state)          # undo any facts the tests added
    print(f"\n{len(tests) - len(failures)}/{len(tests)} tests passed.")
    for desc, r in failures:
        detail = r["message"] if isinstance(r, dict) else r
        print(f"  FAIL: {desc}\n        got: {detail}")
    return failures

_ = run_tests()

Result  Test
PASS    Known light 'Red'
PASS    Case and spacing ' red '
PASS    Space instead of underscore
PASS    Route Nairobi to Kisumu is 363 km
PASS    Route from a town to itself
PASS    Unknown light 'Blue'
PASS    Non-string light value 123
PASS    Python None as obstacle = no obstacle
PASS    Unknown query type 'weather'
PASS    Unknown load cannot depart
PASS    Unknown town
PASS    Empty obstacle reading is not safe
PASS    Obstacle overrides green light
PASS    Overloaded overrides green light
FAIL    Return route Kisumu to Nairobi
PASS    Reject duplicate rule
PASS    New load needs depart_allowed
FAIL    New rule name follows naming style
PASS    Reject zero distance
PASS    Reject negative distance
PASS    Reject text distance
FAIL    Reject NaN distance
FAIL    Reject infinite distance

19/23 tests passed.
  FAIL: Return route Kisumu to Nairobi
        got: No route from kisumu to nairobi exists in the knowledge base.
  FAIL: New rule name follows naming style
        

**9.4 Diagnosis**

The checks and tests reveal three faults in the knowledge base:

| # | Fault | Cause | Effect |
|---|---|---|---|
| 1 | No return route from Kisumu to Nairobi | Every `Road(x, y, d)` fact is one-way, copied from the UCS exercise, which only needed Nairobi to Kisumu | The trailer can deliver cargo but can never return to its depot; most towns cannot reach Nairobi at all |
| 2 | `add_road` accepts `nan` and `inf` distances | The check `d <= 0` is `False` for `nan`, and `inf` is a positive number | A single bad sensor or typing error could break UCS route costs and silently violate Rule 9 |
| 3 | New rule names keep the user's casing (`purple arrow` is stored as `purple_arrow`) | `add_rule` only replaces spaces | Lookups still work because they ignore case, but the stored constants no longer follow the vocabulary naming style (`Flashing_Red`) and the KB display becomes inconsistent |

**9.5 Fixes**

In [ ]:
# Fix 2: reject non-finite distances (Rule 9 requires d > 0 and finite)
_add_road_original = add_road

def add_road(x, y, d, bidirectional=False):
    if isinstance(d, (int, float)) and not isinstance(d, bool) and not math.isfinite(d):
        return _result("rejected", "road", (x, y), None,
                       "Distance must be a finite positive number (Rule 9).")
    return _add_road_original(x, y, d, bidirectional)

# Fix 3: store new rule names in the vocabulary style (Title_Case with underscores)
_add_rule_original = add_rule

def add_rule(kind, key, action, overwrite=False, depart_allowed=None):
    if key is not None and str(key).strip():
        key = "_".join(w.capitalize() for w in _normalise(key).split("_") if w)
    return _add_rule_original(kind, key, action, overwrite, depart_allowed)

# Fix 1: make every road two-way, so the trailer can return to its depot
for x, nbrs in list(knowledge_base["Road"].items()):
    for y, d in list(nbrs.items()):
        if x not in knowledge_base["Road"].get(y, {}):
            add_road(y, x, d)

clean_state = snapshot()     # the fixed KB becomes the new baseline
print("Fixes applied. Road facts now:", sum(len(n) for n in town_graph.values()))

Fixes applied. Road facts now: 26


**9.6 Re-run the checks and tests after the fixes**

In [ ]:
_ = consistency_checks()
print()
_ = run_tests()

print()
show(query_route("kisumu", "nairobi"))
show(query_route("nairobi", "kisumu"))   # the forward route is unchanged

--- Consistency checks ---
  All checks passed.

Result  Test
PASS    Known light 'Red'
PASS    Case and spacing ' red '
PASS    Space instead of underscore
PASS    Route Nairobi to Kisumu is 363 km
PASS    Route from a town to itself
PASS    Unknown light 'Blue'
PASS    Non-string light value 123
PASS    Python None as obstacle = no obstacle
PASS    Unknown query type 'weather'
PASS    Unknown load cannot depart
PASS    Unknown town
PASS    Empty obstacle reading is not safe
PASS    Obstacle overrides green light
PASS    Overloaded overrides green light
PASS    Return route Kisumu to Nairobi
PASS    Reject duplicate rule
PASS    New load needs depart_allowed
PASS    New rule name follows naming style
PASS    Reject zero distance
PASS    Reject negative distance
PASS    Reject text distance
PASS    Reject NaN distance
PASS    Reject infinite distance

23/23 tests passed.

[FOUND] Shortest route: kisumu -> awasi -> kericho -> nakuru -> naivasha -> limuru -> nairobi (363 km).
   Answer: 

**Final Reflection**

**Challenges**

1.  **Choosing a representation.** The base tutorial maps a single percept to a single action with a dictionary. This was enough for traffic lights, but a trailer reacts to several percepts at once, and some rules (for example Rule 6, a steep descent combined with a heavy load) depend on more than one fact. We kept a dictionary for each predicate so that lookups stay simple, and added an explicit priority order and a `decide` function to combine them. This works, but it is a fixed ordering rather than real logical inference.

2.  **Deciding what counts as "safe".** When a percept is missing, unknown or malformed, the agent must still act. We chose to treat every unknown value as dangerous (stop, or refuse to depart), because for a heavy vehicle a needless stop costs far less than a wrong decision to proceed. Choosing these defaults required judgement rather than a clear-cut rule.

3.  **Reusing knowledge from another exercise.** The road network came from the UCS exercise, which only needed to travel from Nairobi to Kisumu. Debugging showed that this assumption had been carried over unnoticed: every road was one-way, so the trailer could never return to its depot. This showed us that knowledge reused from another exercise must be checked against the new task.

4.  **Validating new facts.** Allowing facts to be added at runtime makes the knowledge base flexible but also fragile. The `nan` and `inf` distances passed validation because they satisfy the obvious checks, and would have broken the route costs without raising any error.

**Recommended improvements**

1.  **Use a real inference engine.** Encoding the rules in a forward-chaining system or a logic language (for example Prolog, or a Python library such as `pyDatalog` or `experta`) would let the agent derive conclusions from combinations of facts directly, instead of relying on a hand-written priority list.

2.  **Make route selection knowledge-aware.** UCS currently minimises distance only. Road conditions could be attached to each road fact and used to raise its cost, so that, for example, a loaded trailer prefers a slightly longer route that avoids a steep descent or road works. With straight-line distances between towns as a heuristic, A* search could also reduce the number of nodes expanded.

3.  **Return structured actions.** Actions are currently stored as text meant for a human reader. A real controller needs machine-readable commands (for example `{"brake": True, "target_speed": 40}`) that it can execute and that tests can check precisely.

4.  **Handle uncertainty.** The agent assumes every percept is correct. Real sensors report readings with some confidence, so adding probabilities, or requiring two sensors to agree before an obstacle is cleared, would make decisions more reliable.

5.  **Use real data and persistence.** Road distances could be taken from a routing service such as OpenStreetMap instead of being typed in, and the knowledge base could be saved to a file (for example JSON) so that facts added at runtime are kept between sessions.